# 01 - Load Broad samples
Interface (single / several):
- `load_broad_sample(sample_id) -> AnnData` / `load_broad_samples(sample_ids=None) -> {name: AnnData}` (tar.gz -> unzip -> read -> attach metadata)
- `load_h5ad_sample(sample_id) -> AnnData` / `load_h5ad_samples(sample_ids=None) -> {name: AnnData}` (converted .h5ad)
- `add_gene_info(adata) -> AnnData` (adds `var['symbol', 'chrom', 'is_mt']` from the Ensembl ID)

In [1]:
import sys
sys.path.insert(0, "../src/01_preprocess")  # folder name starts with a digit, so not importable as a package
from cs223_scRNAdataset_read_funs import DATA_DIR, load_broad_sample, load_broad_samples, load_broad_metadata

DATA_DIR  # absolute path, resolved from project root

PosixPath('/Users/solhee/WorkSpace/Human-Cellular-Aging-Agent/data')

## Metadata overview

In [2]:
meta = load_broad_metadata()
meta[["tissue", "age", "sex", "disease_status", "format"]]

,tissue,age,sex,disease_status,format
sample_id,,,,,
Sample_1931,heart,68.0,male,normal,10x 5' v1
Sample_1932,heart,58.0,male,dilated cardiomyopathy,10x 5' v1
Sample_1933,heart,53.0,female,diabetes,10x 5' v1
Sample_1934,heart,62.0,female,diabetes,10x 5' v1
Sample_1935,heart,61.0,female,"dilated cardiomyopathy, diabetes",10x 5' v1
Sample_1936,heart,60.0,male,diabetes,10x 5' v1
Sample_1939,heart,27.0,male,diabetes,10x 5' v1
Sample_1940,heart,44.0,male,normal,10x 5' v1
Sample_1941,heart,9.0,male,dilated cardiomyopathy,10x 5' v1


## One sample (id can be 3444, '3444' or 'Sample_3444')

In [3]:
adata = load_broad_sample(3444)
adata

AnnData object with n_obs × n_vars = 6618 × 25335
    obs: 'barcode', 'sample', 'age', 'sex', 'tissue', 'disease_status', 'study_id', 'format'
    var: 'gene_ids', 'gene_symbols', 'feature_type'
    uns: 'metadata'
    layers: None (.X)

In [4]:
adata.obs.head()

,barcode,sample,age,sex,tissue,disease_status,study_id,format
barcode,,,,,,,,
GCGCAGTTCCAAACTG,GCGCAGTTCCAAACTG,Sample_3444,50.0,female,brain,multiple sclerosis,31316211,10x 3' v2
GTGCAGCCACCCTATC,GTGCAGCCACCCTATC,Sample_3444,50.0,female,brain,multiple sclerosis,31316211,10x 3' v2
GACAGAGCAATTGCTG,GACAGAGCAATTGCTG,Sample_3444,50.0,female,brain,multiple sclerosis,31316211,10x 3' v2
CCCAGTTGTAGCGTGA,CCCAGTTGTAGCGTGA,Sample_3444,50.0,female,brain,multiple sclerosis,31316211,10x 3' v2
GGAAAGCTCAAAGACA,GGAAAGCTCAAAGACA,Sample_3444,50.0,female,brain,multiple sclerosis,31316211,10x 3' v2


In [5]:
adata.var.head()

,gene_ids,gene_symbols,feature_type
gene_symbols,,,
ENSG00000189339,ENSG00000189339,ENSG00000189339,Gene Expression
ENSG00000008130,ENSG00000008130,ENSG00000008130,Gene Expression
ENSG00000078369,ENSG00000078369,ENSG00000078369,Gene Expression
ENSG00000157933,ENSG00000157933,ENSG00000157933,Gene Expression
ENSG00000149527,ENSG00000149527,ENSG00000149527,Gene Expression


In [6]:
adata.uns["metadata"]

{'study_id': 31316211,
 'sample_id': 3444,
 'age': 50.0,
 'cell_types': 'astrocytes, b cells, phagocytes, t cells, endothelial cells, stromal cells, gila cells, oligodendrocyte progenitor cells, neurons',
 'tissue': 'brain',
 'disease_status': 'multiple sclerosis',
 'notes': 'Donor ID is MS8',
 'format': "10x 3' v2",
 'sex': 'female'}

## Several samples -> dict of AnnData

In [7]:
adatas = load_broad_samples([3444, 3443])
{k: v.shape for k, v in adatas.items()}

{'Sample_3444': (6618, 25335), 'Sample_3443': (24241, 26129)}

## Converted .h5ad (after `src/01_preprocess/convert_to_h5ad.py`)
Faster than unpacking the tar every time.

In [8]:
from convert_to_h5ad import load_h5ad_sample, load_h5ad_samples

adata = load_h5ad_sample(1931)
adata

AnnData object with n_obs × n_vars = 8763 × 17680
    obs: 'barcode', 'sample', 'age', 'sex', 'tissue', 'disease_status', 'study_id', 'format'
    var: 'gene_ids', 'gene_symbols', 'feature_type'
    uns: 'metadata'
    layers: None (.X)

In [9]:
adatas = load_h5ad_samples([1931, 1932])
{k: v.shape for k, v in adatas.items()}

{'Sample_1931': (8763, 17680), 'Sample_1932': (5431, 18243)}

## Gene info (Ensembl ID -> symbol / chrom / is_mt)
Needs `data/gene_map.tsv`; if missing, run `python src/01_preprocess/gene_map.py` once.

In [10]:
from gene_map import add_gene_info

add_gene_info(adata)
adata.var[["symbol", "chrom", "is_mt"]].head()

gene map: 16660/17680 genes matched (94.2%), 13 MT genes


,symbol,chrom,is_mt
gene_symbols,,,
ENSG00000171735,CAMTA1,1,False
ENSG00000142599,RERE,1,False
ENSG00000230679,ENO1-AS1,1,False
ENSG00000048707,VPS13D,1,False
ENSG00000189337,KAZN,1,False
